# 03 - Parametric simulation: aPMV setpoints

This notebook runs a parametric study for aPMV setpoints, including output filtering, parameter ranges, and simulation execution.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation`
- `ParametricSimulation.select_outputs`
- `ParametricSimulation.set_parameters`
- `ParametricSimulation.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_output_readers`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## aPMV parametric: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [ ]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.sim import apmv_setpoints
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import ParametricSimulation
from accim.sim.apmv_setpoints import set_pmv_input_parameters

pd.set_option("display.max_columns", 160)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## aPMV parametric: define paths and execution folders

This block defines supporting paths and run constants used by later cells.

Centralizing these values reduces hidden coupling and keeps the execution flow easier to audit.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_file`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_file`, `epw_list`, `RESULTS_DIRNAME`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_file = "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf"
epw_list = ["Seville.epw"]
RESULTS_DIRNAME = "nb03_parametric_apmv_results"

print("IDF exists:", (PROJECT_ROOT / idf_file).exists())
print("EPWs exist:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## aPMV parametric: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [ ]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


## aPMV parametric: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

The PMV that the aPMV setpoints control depends on the inputs of the People objects. The sample model uses a constant schedule of 1 for the work efficiency, the clothing insulation and the air velocity: with a work efficiency of 1, all the metabolic heat becomes work, so the occupants would always feel cold and the thermostats would overheat the zones. `set_pmv_input_parameters` sets a work efficiency of 0 and an air velocity of 0.1 m/s, and a seasonal schedule sets 1 clo from October to March and 0.5 clo from April to September.

`ParametricSimulation` applies the aPMV setpoints to the model when it is created (`parameters_type="apmv setpoints"`). `add_vrf_system` has already added a thermal comfort thermostat to each zone, and the model already requests some of the hourly outputs of the aPMV setpoints, so the warnings about a thermostat or an output variable that already exists are expected: the thermostats are set up for the aPMV setpoints and the repeated outputs are skipped.

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`, `parametric`.

In [ ]:
building = ef.get_building(idf_file)
accim.utils.reduce_runtime(
    idf_object=building,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"
# Ensure the model contains the HVAC structure expected by aPMV routines.
apmv_setpoints.add_vrf_system(building=building)

# PMV inputs of the occupants: the sample model uses 1 for the work efficiency, the clothing
# insulation (clo) and the air velocity (m/s), which overheats the zones under aPMV control.
set_pmv_input_parameters(building, air_velocity=0.1, work_efficiency=0.0, verbose_mode=False)
building.newidfobject(
    "Schedule:Compact",
    Name="Clothing insulation",
    Schedule_Type_Limits_Name="Any Number",
    Field_1="Through: 31 Mar", Field_2="For: AllDays", Field_3="Until: 24:00, 1.0",
    Field_4="Through: 30 Sep", Field_5="For: AllDays", Field_6="Until: 24:00, 0.5",
    Field_7="Through: 31 Dec", Field_8="For: AllDays", Field_9="Until: 24:00, 1.0",
)
for people in building.idfobjects["People"]:
    people.Clothing_Insulation_Schedule_Name = "Clothing insulation"

parametric = ParametricSimulation(
	building=building,
	parameters_type="apmv setpoints",
	epws=epw_list,
	output_freqs=["hourly"],
)


## aPMV parametric: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.
- `meters`: energy meters to include as outputs.
- `variables`: output variables to include by name or pattern.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.
- `report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:
available_outputs = parametric.discover_available_outputs(
	reduce_sim_time=True,
	prefer="testsimeplus",
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]
print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))

selection = parametric.select_outputs(
	meters=["Heating:Electricity", "Cooling:Electricity"],
	variables=["aPMV"],
	match="contains",
	on_missing="warn",
	suggest=True,
)
df_meters_sel = selection["meters"]
df_vars_sel = selection["variables"]
report = selection["report"]
print("Missing outputs:", report.get("missing", {}))

parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=df_vars_sel,
	clean_mode="all",
	validate_before_apply=False,
	validate_after_apply=True,
	on_missing="warn",
)
parametric.set_output_readers(df_output_meter=df_meters_sel)


## aPMV parametric: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.

Variables created in this cell:
- `apmv_parameters`: reused in subsequent workflow blocks.
- `plan_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `apmv_parameters`, `plan_df`.

In [ ]:
apmv_parameters = {
	"Adaptive coefficient": [0.0, 0.9],
	"PMV setpoint": [0.2, 0.8],
}
parametric.set_parameters(accis_params_dict=apmv_parameters)
parametric.set_problem()
parametric.sampling_full_set()

plan_df = parametric.parameters_values_df.copy()
print("Plan rows:", len(plan_df))
print(plan_df.head())


## aPMV parametric: estimate case count from plan and EPW multipliers

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

Variables created in this cell:
- `epw_multiplier`: reused in subsequent workflow blocks.
- `estimated_simulations`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
epw_multiplier = 1 if "epw" in plan_df.columns else len(epw_list)
estimated_simulations = len(plan_df) * epw_multiplier

print("EPW multiplier:", epw_multiplier)
print("Estimated simulation cases:", estimated_simulations)


## aPMV parametric: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [ ]:

outputs = parametric.run_parametric_simulation(
    out_dir=RESULTS_DIRNAME,
    df=plan_df,
    processes=PROCESSES,
    keep_input=True,
    keep_dirs=False,
)
print("Rows:", len(outputs))
print(outputs.head())


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## aPMV parametric: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [ ]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


## aPMV parametric: remove generated artifacts from this run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
